In [65]:
import os
from datetime import datetime, timezone

import pandas as pd
import requests
from dotenv import load_dotenv

import domain

In [12]:
load_dotenv()

API_TOKEN = os.environ.get("CLASH_API_TOKEN")
if not API_TOKEN:
    raise RuntimeError("Falta CLASH_API_KEY no .env / env vars.")

HEADERS = {"Authorization": f"Bearer {API_TOKEN}"}

def cr_get(url: str):
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

In [13]:
def fetch_clan_members(clan_tag_no_hash: str) -> list[dict]:
    url = f"https://api.clashroyale.com/v1/clans/%23{clan_tag_no_hash}/members"
    data = cr_get(url)
    return data.get("items", data)

def fetch_player_battlelog(player_tag_no_hash: str) -> list[dict]:
    url = f"https://api.clashroyale.com/v1/players/%23{player_tag_no_hash}/battlelog"
    return cr_get(url)

CLAN_TAG = "QGVYLRVG"  # sem #
members = fetch_clan_members(CLAN_TAG)

# battlelogs em memória (dict: tag -> list[battles])
battles_by_player = {}
for m in members:
    tag_no_hash = m["tag"].replace("#", "")
    battles_by_player[tag_no_hash] = fetch_player_battlelog(tag_no_hash)

len(members), len(battles_by_player)


(49, 49)

In [75]:
from domain.models.player import Player
from domain.models.battle import Battle
from domain.filters.battle_filter import battle_weight

[autoreload of domain.scoring.activity_score failed: Traceback (most recent call last):
  File "C:\Users\PC Multimedia\anaconda3\envs\crownledger-bot\Lib\site-packages\IPython\extensions\autoreload.py", line 322, in check
    elif self.deduper_reloader.maybe_reload_module(m):
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\PC Multimedia\anaconda3\envs\crownledger-bot\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 545, in maybe_reload_module
    new_source_code = f.read()
                      ^^^^^^^^
  File "C:\Users\PC Multimedia\anaconda3\envs\crownledger-bot\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 3436: character maps to <undefined>
]


In [76]:
def normalize_tag(tag: str) -> str:
    return (tag or "").replace("#", "").strip()

def parse_battle_time_utc(b: dict) -> datetime:
    """
    Clash Royale battleTime típico: 'YYYYMMDDThhmmss.000Z'
    Ex: '20251220T212548.000Z'
    """
    s = (b.get("battleTime") or "").strip()
    if not s:
        return datetime.fromtimestamp(0, tz=timezone.utc)

    # Formato CR comum: 20251220T212548.000Z
    if len(s) >= 15 and s[8] == "T":
        iso = f"{s[0:4]}-{s[4:6]}-{s[6:8]}T{s[9:11]}:{s[11:13]}:{s[13:15]}"
        tail = s[15:]  # pode incluir ".000Z"
        if tail:
            iso += tail
        return datetime.fromisoformat(iso.replace("Z", "+00:00"))

    # fallback
    return datetime.fromisoformat(s.replace("Z", "+00:00"))


In [77]:
players: list[Player] = []

for m in members:
    tag_raw = m.get("tag", "")
    name = m.get("name", "")

    p = Player(tag=tag_raw, name=name)

    tag_no_hash = normalize_tag(tag_raw)

    # tenta buscar tanto por "#TAG" como por "TAG"
    raw_battles = (
        battles_by_player.get(tag_no_hash)
        or battles_by_player.get(tag_raw)
        or []
    )

    # cria Battles para o teu Player (o resto do pipeline fica no domain)
    p.battles = [
        Battle(
            timestamp=parse_battle_time_utc(b),
            battle_type=(b.get("type") or b.get("battleType") or ""),
            raw_json=b
        )
        for b in raw_battles
    ]

    players.append(p)

len(players), sum(len(p.battles) for p in players)


(49, 1653)

In [69]:
from domain.scoring.activity_score import compute_clan_baseline

snapshots = [p.activity_snapshot() for p in players]
baseline = compute_clan_baseline(snapshots, max_battles=40, max_days=15)

rows = []
for p in players:
    snap = p.activity_snapshot()
    score = p.activity_score()

    eff_count = sum(1 for b in p.battles if battle_weight(b.raw_json) > 0.0)

    rows.append({
        "name": p.name,
        "tag": p.tag.replace("#", ""),
        "score": p.activity_score(clan_baseline=baseline),
        "days_since_last_effective": snap["days_since_last_effective"],
        "days_with_battles": snap["days_with_battles"],
        "raw_7d": snap.get("raw_7d"),
        "effective_7d": snap.get("effective_7d"),
        "weighted_7d": snap.get("weighted_7d"),
        "weighted_2d": snap.get("weighted_2d"),

        # sanity checks úteis
        "battles_total_fetched": len(p.battles),
        "effective_fetched": eff_count,
        "has_effective": eff_count > 0,
    })

df = pd.DataFrame(rows)

# ordenação: quem tem efetivas primeiro, depois score
df = df.sort_values(["has_effective", "score", "weighted_7d", "weighted_2d"], ascending=[False, False, False, False]).reset_index(drop=True)
df.insert(0, "rank", df.index + 1)

# opcional: não mostrar a coluna auxiliar
df.drop(columns=["has_effective"], inplace=True)

df.head(50)


,rank,name,tag,score,days_since_last_effective,days_with_battles,raw_7d,effective_7d,weighted_7d,weighted_2d,battles_total_fetched,effective_fetched
0,1,sssantos,RRJ228L9J,0.738,0.066451,3,31,29,29.0,29.0,31,29
1,2,RibeiroGamer,UL9YR09Q,0.723,0.428616,2,30,30,30.0,27.0,30,30
2,3,Ruizao,UG9LPJUP,0.721,0.498986,3,30,30,30.0,27.0,36,30
3,4,ragazini,U8U0LURY,0.700,0.067239,8,30,30,30.0,20.0,40,30
4,5,Rapanui,CJQ0J8JJV,0.681,0.553824,6,33,25,25.0,25.0,40,25
5,6,SalvaSLB,98R2U0R9,0.674,0.089854,7,35,30,30.0,15.0,40,30
6,7,#richa#,LVPQPRPP,0.668,0.100745,4,32,30,30.0,14.0,32,30
7,8,velhacometudo,PV9V99LJG,0.668,0.182296,5,30,27,27.0,18.0,40,27
8,9,Darkgaia,QGUV08UC,0.665,0.197898,4,30,30,30.0,14.0,30,30
9,10,k3vin35,2GQRYQLRR,0.659,0.362759,3,30,23,23.0,22.0,30,23


In [71]:
weights = []
for p in players:
    for b in p.battles:
        weights.append(battle_weight(b.raw_json))

pd.Series(weights).value_counts().sort_index()


0.0     530
1.0    1123
Name: count, dtype: int64

In [78]:
import pandas as pd
from domain.scoring.activity_score import compute_clan_baseline, compute_activity_score

snapshots = [p.activity_snapshot() for p in players]
baseline = compute_clan_baseline(snapshots)

rows = []
for p in players:
    s = p.activity_snapshot()
    rows.append({
        "name": p.name,
        "tag": p.tag.replace("#", ""),
        "score": compute_activity_score(s, clan_baseline=baseline),
        "weighted_7d": s["weighted_7d"],
        "weighted_2d": s["weighted_2d"],
        "days_since_last_effective": s["days_since_last_effective"],
        "battles_total_fetched": s["battles_total_fetched"],
        "days_with_battles": s["days_with_battles"],
        "effective_7d": s["effective_7d"],
        "raw_7d": s["raw_7d"],
    })

df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
df.insert(0, "rank", df.index + 1)
df


,rank,name,tag,score,weighted_7d,weighted_2d,days_since_last_effective,battles_total_fetched,days_with_battles,effective_7d,raw_7d
0,1,Ruizao,UG9LPJUP,0.748,30.0,27.0,0.526134,36,3,30,30
1,2,RibeiroGamer,UL9YR09Q,0.733,30.0,27.0,0.455764,30,2,30,30
2,3,sssantos,RRJ228L9J,0.709,29.0,29.0,0.093599,31,3,29,31
3,4,ragazini,U8U0LURY,0.707,30.0,20.0,0.094386,40,8,30,30
4,5,Rapanui,CJQ0J8JJV,0.705,25.0,25.0,0.580972,40,6,25,33
5,6,velhacometudo,PV9V99LJG,0.700,27.0,16.0,0.209444,40,5,27,30
6,7,SalvaSLB,98R2U0R9,0.694,30.0,15.0,0.117002,40,7,30,35
7,8,Dudas,28282PRLP,0.689,21.0,21.0,1.004653,40,4,21,30
8,9,carapau,98U8Q0V0R,0.672,30.0,11.0,0.380359,40,7,30,30
9,10,carecagostosO,LLQPCJ28V,0.671,30.0,8.0,0.274896,40,6,30,30
